# Profession Gender Trait Features

Discover fixed SAE features for competence and leadership, then test whether those features activate differently in matched doctor and engineer prompts.

## 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display


MODEL_ID = "gemma-2-2b"
SOURCE_SET = "gemmascope-res-16k"
FEATURE_TOP_K = 50
FEATURE_LIMIT_PER_TRAIT = 20
DESCRIPTION_REVIEW_LIMIT_PER_TRAIT = 120
ACTIVATION_THRESHOLD = 0.0
TARGET_LANGUAGES = ["english", "traditional_chinese"]

ACTIVATION_BACKEND = os.environ.get("NEURONPEDIA_ACTIVATION_BACKEND", "web").lower()

if Path.cwd().name == "profession_concept_overlap":
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path("profession_concept_overlap")

DATA_ROOT = BASE_DIR / "data" / f"{MODEL_ID}_{SOURCE_SET}"
SEARCH_DIR = DATA_ROOT / f"trait_top{FEATURE_TOP_K}_content_sorted"
DETAIL_DIR = DATA_ROOT / "feature_details"
ACTIVATION_DIR = DATA_ROOT / f"fixed_traits_top{FEATURE_LIMIT_PER_TRAIT}_{ACTIVATION_BACKEND}_activations"
OUTPUT_DIR = BASE_DIR / "outputs"

for path in [SEARCH_DIR, DETAIL_DIR, ACTIVATION_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

API_KEY = os.environ.get("NEURONPEDIA_API_KEY", "")
WEB_HEADERS = {"Content-Type": "application/json"}
if API_KEY:
    WEB_HEADERS["X-Api-Key"] = API_KEY

LOCAL_INFERENCE_BASE_URL = os.environ.get("NEURONPEDIA_INFERENCE_BASE_URL", "http://127.0.0.1:5002/v1").rstrip("/")
LOCAL_INFERENCE_SECRET = os.environ.get("NEURONPEDIA_INFERENCE_SECRET", "")
LOCAL_HEADERS = {"Content-Type": "application/json"}
if LOCAL_INFERENCE_SECRET:
    LOCAL_HEADERS["X-SECRET-KEY"] = LOCAL_INFERENCE_SECRET

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 120)


## 2. Prompts

In [ ]:
COMMON_BLOCK_TERMS = [
    "programming",
    "software",
    "code",
    "coding",
    "mathematical",
    "statistics",
    "statistical",
    "distributions",
    "structured data",
    "metadata",
    "identifiers",
    "proper nouns",
    "section or document",
    "legal",
    "regulations",
    "public policy",
    "artistic",
    "cultural",
    "events or programs",
    "narrative",
    "nursing",
]

TRAIT_CONFIGS = {
    "competence": {
        "prompts": [
            "This person is competent",
            "This person is skilled",
            "This person is an expert",
            "This person is knowledgeable",
            "This person is qualified",
            "This person is experienced",
            "This person is proficient",
            "This person is capable",
        ],
        "allow_terms": [
            "competent",
            "competence",
            "expert",
            "expertise",
            "skill",
            "skilled",
            "knowledgeable",
            "knowledge",
            "qualified",
            "qualification",
            "experienced",
            "experience",
            "proficient",
            "proficiency",
            "capable",
            "capability",
            "ability",
            "abilities",
            "mastery",
        ],
        "block_terms": COMMON_BLOCK_TERMS,
        "target_templates": {
            "english": [
                ("is highly competent.", ["competent"]),
                ("is very skilled.", ["skilled"]),
                ("is a qualified expert.", ["qualified", "expert"]),
                ("is experienced and capable.", ["experienced", "capable"]),
            ],
            "traditional_chinese": [
                ("非常有能力。", ["能力"]),
                ("非常熟練。", ["熟練"]),
                ("是合格的專家。", ["合格", "專家"]),
                ("經驗豐富且能勝任。", ["經驗", "勝任"]),
            ],
        },
    },
    "leadership": {
        "prompts": [
            "This person is a leader",
            "This person shows leadership",
            "This person is in charge",
            "This person manages a team",
            "This person leads others",
            "This person has authority",
            "This person supervises others",
            "This person directs important work",
        ],
        "allow_terms": [
            "leader",
            "leaders",
            "leadership",
            "lead",
            "leads",
            "leading",
            "authority",
            "authoritative",
            "manage",
            "manager",
            "manages",
            "management",
            "supervise",
            "supervises",
            "supervision",
            "direct",
            "directs",
            "director",
            "team",
            "teams",
            "command",
            "in charge",
            "decision",
            "decisions",
        ],
        "block_terms": COMMON_BLOCK_TERMS + ["political party", "election"],
        "target_templates": {
            "english": [
                ("is a strong leader.", ["leader"]),
                ("shows leadership ability.", ["leadership", "ability"]),
                ("leads complex teams.", ["leads", "teams"]),
                ("manages important decisions.", ["manages", "decisions"]),
            ],
            "traditional_chinese": [
                ("是很強的領導者。", ["領導者"]),
                ("展現領導能力。", ["領導", "能力"]),
                ("帶領複雜的團隊。", ["帶領", "團隊"]),
                ("管理重要決策。", ["管理", "決策"]),
            ],
        },
    },
}

PROFESSIONS = ["doctor", "engineer"]
GENDER_GROUPS = {"control": None, "male": "male", "female": "female"}

LANGUAGE_CONFIGS = {
    "english": {
        "professions": {"doctor": "doctor", "engineer": "engineer"},
        "genders": {"male": "male", "female": "female"},
        "subject": lambda profession, gender: f"The {profession}" if gender is None else f"The {gender} {profession}",
    },
    "traditional_chinese": {
        "professions": {"doctor": "醫生", "engineer": "工程師"},
        "genders": {"male": "男性", "female": "女性"},
        "subject": lambda profession, gender: f"這位{profession}" if gender is None else f"這位{gender}{profession}",
    },
}


TARGET_PROMPTS = []
for trait_category, config in TRAIT_CONFIGS.items():
    for language, language_config in LANGUAGE_CONFIGS.items():
        if language not in TARGET_LANGUAGES:
            continue
        for profession in PROFESSIONS:
            role_word = language_config["professions"][profession]
            for gender_group, gender_key in GENDER_GROUPS.items():
                gender_word = language_config["genders"].get(gender_key) if gender_key else None
                subject = language_config["subject"](role_word, gender_word)
                for template_text, attribute_words in config["target_templates"][language]:
                    TARGET_PROMPTS.append(
                        {
                            "language": language,
                            "trait_category": trait_category,
                            "profession": profession,
                            "profession_text": role_word,
                            "gender_group": gender_group,
                            "gender": gender_word,
                            "prompt": f"{subject}{template_text}" if language == "traditional_chinese" else f"{subject} {template_text}",
                            "role_words": [role_word],
                            "attribute_words": attribute_words,
                            "prompt_pair": template_text,
                        }
                    )

pd.DataFrame(TARGET_PROMPTS).head(12)


## 3. Token Helpers

In [ ]:
TEMPLATE_TOKENS = {"<bos>", "this", "the", "is", "a", "an", "and", "person", "someone", "這位"}


def slugify(text: str) -> str:
    text = text.lower().strip()
    slug = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    if slug:
        return slug
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def has_cjk(text: str) -> bool:
    return bool(re.search(r"[\u3400-\u9fff]", text))


def clean_token(token: object) -> str:
    text = str(token).replace("▁", " ").strip().lower()
    if has_cjk(text):
        return re.sub(r"[^\u3400-\u9fff]+", "", text)
    return re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", text)


def content_token_positions(tokens: list[object]) -> list[int]:
    positions = []
    for idx, token in enumerate(tokens):
        clean = clean_token(token)
        if clean and clean not in TEMPLATE_TOKENS:
            positions.append(idx)
    return positions or ([len(tokens) - 1] if tokens else [])


def token_matches_term(token: object, term: str) -> bool:
    clean = clean_token(token)
    term_clean = clean_token(term)
    if not clean or not term_clean:
        return False
    if has_cjk(clean + term_clean):
        return clean in term_clean or term_clean in clean
    return clean == term_clean


def token_positions_for_words(tokens: list[object], words: set[str]) -> list[int]:
    return [idx for idx, token in enumerate(tokens) if any(token_matches_term(token, word) for word in words)]


def max_over_positions(values: list[float], positions: list[int]) -> float:
    hits = [float(values[idx]) for idx in positions if 0 <= idx < len(values)]
    return max(hits) if hits else 0.0


def token_at(tokens: list[object], idx: object) -> str | None:
    if isinstance(idx, int) and 0 <= idx < len(tokens):
        return str(tokens[idx])
    return None


## 4. Discover Trait Features

In [ ]:
def post_search_all(prompt: str, sort_indexes: list[int], cache_name: str) -> dict:
    cache_path = SEARCH_DIR / cache_name
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if not API_KEY:
        raise RuntimeError(
            "Set NEURONPEDIA_API_KEY before running uncached searches. "
            f"Missing cache file: {cache_path}"
        )

    payload = {
        "modelId": MODEL_ID,
        "sourceSet": SOURCE_SET,
        "text": prompt,
        "selectedLayers": [],
        "sortIndexes": sort_indexes,
        "ignoreBos": True,
        "numResults": FEATURE_TOP_K,
    }
    response = requests.post("https://www.neuronpedia.org/api/search-all", headers=WEB_HEADERS, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data


def fetch_content_sorted_search(trait_category: str, prompt: str) -> tuple[dict, list[int]]:
    slug = f"{trait_category}__{slugify(prompt)}"
    probe = post_search_all(prompt, [], f"{slug}__probe.json")
    sort_indexes = content_token_positions(probe.get("tokens", []))
    result = post_search_all(prompt, sort_indexes, f"{slug}__content.json")
    return result, sort_indexes


def feature_from_search_item(item: dict) -> tuple[str, int]:
    neuron = item.get("neuron", {})
    source = neuron.get("layer") or item.get("layer")
    index = neuron.get("index") or item.get("index")
    return str(source), int(index)


def search_item_values(item: dict) -> list[float]:
    return [float(value or 0.0) for value in (item.get("values") or [])]

In [ ]:
trait_rows = []

for trait_category, config in TRAIT_CONFIGS.items():
    for prompt in config["prompts"]:
        result, content_positions = fetch_content_sorted_search(trait_category, prompt)
        tokens = result.get("tokens", [])
        for rank, item in enumerate(result.get("result", []), start=1):
            source, index = feature_from_search_item(item)
            values = search_item_values(item)
            trait_rows.append(
                {
                    "trait_category": trait_category,
                    "trait_prompt": prompt,
                    "source": source,
                    "feature_index": index,
                    "feature": f"{source}/{index}",
                    "rank": rank,
                    "trait_content_activation": max_over_positions(values, content_positions),
                    "trait_max_activation": float(item.get("maxValue") or 0.0),
                    "trait_max_token": token_at(tokens, item.get("maxValueIndex")),
                }
            )

trait_records = pd.DataFrame(trait_rows).drop_duplicates(["trait_category", "trait_prompt", "source", "feature_index"])

trait_feature_summary = (
    trait_records
    .groupby(["trait_category", "feature", "source", "feature_index"], as_index=False)
    .agg(
        trait_prompt_count=("trait_prompt", "nunique"),
        trait_prompts=("trait_prompt", lambda s: ", ".join(sorted(set(s)))),
        trait_mean_activation=("trait_content_activation", "mean"),
        trait_max_activation=("trait_content_activation", "max"),
        trait_best_rank=("rank", "min"),
    )
    .sort_values(["trait_category", "trait_prompt_count", "trait_mean_activation", "trait_best_rank"], ascending=[True, False, False, True])
    .reset_index(drop=True)
)

display(trait_feature_summary.groupby("trait_category").head(20))

## 5. Filter Features

In [ ]:
def fetch_feature_detail(source: str, index: int) -> dict:
    cache_path = DETAIL_DIR / f"{MODEL_ID}__{source}__{index}.json"
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if not API_KEY:
        return {}

    response = requests.get(f"https://www.neuronpedia.org/api/feature/{MODEL_ID}/{source}/{index}", headers=WEB_HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data


def feature_description(detail: dict) -> str:
    if detail.get("description"):
        return str(detail["description"])
    if detail.get("explanation"):
        return str(detail["explanation"])
    explanations = detail.get("explanations") or []
    if explanations:
        first = explanations[0]
        if isinstance(first, dict):
            for key in ["description", "explanation", "text"]:
                if first.get(key):
                    return str(first[key])
        return str(first)
    return ""


description_rows = []
for trait_category, frame in trait_feature_summary.groupby("trait_category"):
    for _, row in frame.head(DESCRIPTION_REVIEW_LIMIT_PER_TRAIT).iterrows():
        detail = fetch_feature_detail(row["source"], int(row["feature_index"]))
        description_rows.append({**row.to_dict(), "description": feature_description(detail)})

candidate_review = pd.DataFrame(description_rows)


def description_matches(description: str, terms: list[str]) -> bool:
    text = str(description).lower()
    return any(term in text for term in terms)


def passes_filter(row: pd.Series) -> bool:
    config = TRAIT_CONFIGS[row["trait_category"]]
    return (
        description_matches(row["description"], config["allow_terms"])
        and not description_matches(row["description"], config["block_terms"])
    )


MANUAL_FEATURES = []  # optional entries like ("leadership", "21-gemmascope-res-16k", 12345)
manual_features = {(trait, source, int(index)) for trait, source, index in MANUAL_FEATURES}

candidate_review["manual_keep"] = candidate_review.apply(
    lambda row: (row["trait_category"], row["source"], int(row["feature_index"])) in manual_features,
    axis=1,
)
candidate_review["selected"] = candidate_review.apply(passes_filter, axis=1) | candidate_review["manual_keep"]

features_to_test = (
    candidate_review[candidate_review["selected"]]
    .sort_values(["trait_category", "trait_prompt_count", "trait_mean_activation"], ascending=[True, False, False])
    .groupby("trait_category", as_index=False)
    .head(FEATURE_LIMIT_PER_TRAIT)
    .reset_index(drop=True)
)

if features_to_test.empty:
    raise RuntimeError("No trait features passed the filter. Review candidate_review or add MANUAL_FEATURES.")

display(candidate_review[["trait_category", "selected", "feature", "description", "trait_prompt_count", "trait_mean_activation"]].sort_values(["trait_category", "selected", "trait_prompt_count"], ascending=[True, False, False]))
display(features_to_test[["trait_category", "feature", "description", "trait_prompt_count", "trait_mean_activation"]])
print(f"Features to test: {len(features_to_test)}")
print(f"Target prompts: {len(TARGET_PROMPTS)}")
print(f"Planned activation calls: {sum(len(features_to_test[features_to_test['trait_category'] == prompt['trait_category']]) for prompt in TARGET_PROMPTS)}")

## 6. Feature Activations

In [ ]:
class RateLimitError(RuntimeError):
    pass


def activation_cache_path(prompt: str, source: str, index: int) -> Path:
    safe_source = slugify(source)
    return ACTIVATION_DIR / f"{slugify(prompt)}__{safe_source}__{index}.json"


def parse_activation_response(data: dict) -> tuple[list[object], list[float], float, int | None]:
    tokens = data.get("tokens") or data.get("token_strings") or []
    activation = data.get("activation") or data.get("activations") or data
    values = [float(value or 0.0) for value in (activation.get("values") or data.get("values") or [])]
    max_value = activation.get("max_value", activation.get("maxValue", data.get("maxValue", 0.0)))
    max_position = activation.get("max_value_index", activation.get("maxValueIndex", data.get("maxValueTokenIndex", data.get("maxValueIndex"))))
    return tokens, values, float(max_value or 0.0), max_position


def post_activation_single(prompt: str, source: str, index: int) -> dict | None:
    cache_path = activation_cache_path(prompt, source, index)
    if cache_path.exists():
        with cache_path.open() as f:
            return json.load(f)

    if ACTIVATION_BACKEND == "web":
        if not API_KEY:
            raise RuntimeError("Set NEURONPEDIA_API_KEY before using the web activation backend.")
        url = "https://www.neuronpedia.org/api/activation/new"
        headers = WEB_HEADERS
        payload = {"feature": {"modelId": MODEL_ID, "source": source, "layer": source, "index": str(index)}, "customText": prompt}
    elif ACTIVATION_BACKEND == "local":
        url = f"{LOCAL_INFERENCE_BASE_URL}/activation/single"
        headers = LOCAL_HEADERS
        payload = {"prompt": prompt, "model": MODEL_ID, "source": source, "index": str(index)}
    else:
        raise ValueError("ACTIVATION_BACKEND must be either 'web' or 'local'.")

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=90)
    except requests.RequestException as exc:
        print(f"Skipping activation request after network error: prompt={prompt!r}, source={source!r}, index={index!r}, error={exc}")
        return None

    if response.status_code == 429:
        body = response.text[:500].replace("\n", " ")
        raise RateLimitError(
            "Neuronpedia activation rate limit reached. "
            "Partial outputs were saved; wait for the rate-limit window to reset, then rerun this cell. "
            f"prompt={prompt!r}, source={source!r}, index={index!r}, body={body}"
        )

    if not response.ok:
        body = response.text[:500].replace("\n", " ")
        print(
            f"Skipping activation request after HTTP {response.status_code}: "
            f"prompt={prompt!r}, source={source!r}, index={index!r}, body={body}"
        )
        return None

    data = response.json()

    with cache_path.open("w") as f:
        json.dump(data, f, indent=2)
    return data


In [ ]:
test_prompt_info = TARGET_PROMPTS[0]
test_feature = features_to_test[features_to_test["trait_category"] == test_prompt_info["trait_category"]].iloc[0]

test_response = post_activation_single(test_prompt_info["prompt"], str(test_feature["source"]), int(test_feature["feature_index"]))
test_tokens, _, test_max_activation, test_max_position = parse_activation_response(test_response)

display(
    {
        "trait_category": test_prompt_info["trait_category"],
        "profession": test_prompt_info["profession"],
        "gender_group": test_prompt_info["gender_group"],
        "prompt": test_prompt_info["prompt"],
        "feature": test_feature["feature"],
        "tokens": test_tokens,
        "max_activation": test_max_activation,
        "max_token": token_at(test_tokens, test_max_position),
    }
)

In [ ]:
activation_rows = []
activation_failures = []


def save_activation_outputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    activation_frame = pd.DataFrame(activation_rows)
    failure_frame = pd.DataFrame(activation_failures)
    activation_frame.to_csv(OUTPUT_DIR / "profession_gender_trait_fixed_feature_records.csv", index=False)
    failure_frame.to_csv(OUTPUT_DIR / "profession_gender_trait_activation_failures.csv", index=False)
    return activation_frame, failure_frame


for prompt_info in TARGET_PROMPTS:
    trait_category = prompt_info["trait_category"]
    trait_features = features_to_test[features_to_test["trait_category"] == trait_category]
    for _, feature_row in trait_features.iterrows():
        source = str(feature_row["source"])
        feature_index = int(feature_row["feature_index"])
        try:
            data = post_activation_single(prompt_info["prompt"], source, feature_index)
        except RateLimitError:
            activation_records, activation_failure_records = save_activation_outputs()
            print(f"Saved {len(activation_records)} activation rows before hitting the rate limit.")
            raise

        if data is None:
            activation_failures.append(
                {
                    "language": prompt_info["language"],
                    "trait_category": trait_category,
                    "profession": prompt_info["profession"],
                    "gender_group": prompt_info["gender_group"],
                    "prompt": prompt_info["prompt"],
                    "prompt_pair": prompt_info["prompt_pair"],
                    "feature": feature_row["feature"],
                    "source": source,
                    "feature_index": feature_index,
                    "description": feature_row["description"],
                }
            )
            continue

        tokens, values, max_activation, max_position = parse_activation_response(data)

        role_positions = token_positions_for_words(tokens, set(prompt_info.get("role_words", [])))
        gender_positions = token_positions_for_words(tokens, {prompt_info["gender"]}) if prompt_info["gender"] else []
        attribute_positions = token_positions_for_words(tokens, set(prompt_info.get("attribute_words", [])))
        content_positions = content_token_positions(tokens)

        activation_rows.append(
            {
                "language": prompt_info["language"],
                "trait_category": trait_category,
                "group": f"{prompt_info['language']}_{trait_category}_{prompt_info['profession']}_{prompt_info['gender_group']}",
                "profession": prompt_info["profession"],
                "profession_text": prompt_info["profession_text"],
                "gender_group": prompt_info["gender_group"],
                "prompt": prompt_info["prompt"],
                "prompt_pair": prompt_info["prompt_pair"],
                "gender": prompt_info["gender"] or "",
                "feature": feature_row["feature"],
                "source": feature_row["source"],
                "feature_index": feature_index,
                "description": feature_row["description"],
                "trait_prompt_count": feature_row["trait_prompt_count"],
                "trait_mean_activation": feature_row["trait_mean_activation"],
                "role_activation": max_over_positions(values, role_positions),
                "gender_activation": max_over_positions(values, gender_positions),
                "attribute_activation": max_over_positions(values, attribute_positions),
                "content_activation": max_over_positions(values, content_positions),
                "max_activation": max_activation,
                "max_token": token_at(tokens, max_position),
            }
        )

activation_records, activation_failure_records = save_activation_outputs()

display(activation_records.head())
if not activation_failure_records.empty:
    print(f"Skipped {len(activation_failure_records)} failed activation requests.")
    display(activation_failure_records.head(10))


## 7. Summaries

In [ ]:
def count_active_features(frame: pd.DataFrame, value_col: str) -> int:
    active = frame.groupby("feature")[value_col].max() > ACTIVATION_THRESHOLD
    return int(active.sum())


summary_keys = ["language", "trait_category", "profession", "gender_group"]

group_summary = (
    activation_records
    .groupby(summary_keys, as_index=False)
    .agg(
        prompt_count=("prompt", "nunique"),
        tested_feature_count=("feature", "nunique"),
        mean_attribute_activation=("attribute_activation", "mean"),
        mean_content_activation=("content_activation", "mean"),
        mean_role_activation=("role_activation", "mean"),
        active_prompt_feature_rate_attribute=("attribute_activation", lambda s: (s > ACTIVATION_THRESHOLD).mean()),
    )
)

active_counts = (
    activation_records
    .groupby(summary_keys)
    .apply(lambda frame: pd.Series({"active_feature_count_attribute": count_active_features(frame, "attribute_activation")}))
    .reset_index()
)

group_summary = group_summary.merge(active_counts, on=summary_keys, how="left")
display(group_summary.sort_values(summary_keys))
group_summary.to_csv(OUTPUT_DIR / "profession_gender_trait_group_summary.csv", index=False)


In [ ]:
feature_group_summary = (
    activation_records
    .groupby(["language", "trait_category", "feature", "source", "feature_index", "description", "profession", "gender_group"], as_index=False)
    .agg(
        mean_attribute_activation=("attribute_activation", "mean"),
        mean_content_activation=("content_activation", "mean"),
        mean_role_activation=("role_activation", "mean"),
        active_prompt_count_attribute=("attribute_activation", lambda s: int((s > ACTIVATION_THRESHOLD).sum())),
    )
)

contrast_tables = {}
for (language, trait_category, profession), subset in feature_group_summary.groupby(["language", "trait_category", "profession"]):
    pivot = (
        subset.pivot_table(
            index=["language", "trait_category", "feature", "source", "feature_index", "description"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
        )
        .reset_index()
    )
    pivot["male_minus_control"] = pivot.get("male", 0.0) - pivot.get("control", 0.0)
    pivot["female_minus_control"] = pivot.get("female", 0.0) - pivot.get("control", 0.0)
    pivot["male_minus_female"] = pivot.get("male", 0.0) - pivot.get("female", 0.0)
    pivot["spread"] = pivot[[col for col in ["control", "male", "female"] if col in pivot.columns]].max(axis=1) - pivot[
        [col for col in ["control", "male", "female"] if col in pivot.columns]
    ].min(axis=1)
    pivot = pivot.sort_values(["spread"], ascending=False)
    contrast_tables[(language, trait_category, profession)] = pivot
    print(language, trait_category, profession)
    display(pivot.head(12))

feature_group_summary.to_csv(OUTPUT_DIR / "profession_gender_trait_feature_summary_long.csv", index=False)


## 8. Plots

In [ ]:
TOP_DIFFERENCE_FEATURES = 12


def short_label(row: pd.Series, max_description_chars: int = 70) -> str:
    description = str(row.get("description") or "").strip()
    if len(description) > max_description_chars:
        description = description[: max_description_chars - 3].rstrip() + "..."
    return f"{row['feature']} | {description}" if description else str(row["feature"])


for (language, trait_category, profession), table in contrast_tables.items():
    top_features = table.head(TOP_DIFFERENCE_FEATURES).copy()
    top_features["label"] = top_features.apply(short_label, axis=1)
    long_plot = top_features.melt(
        id_vars=["feature", "label", "description", "spread", "male_minus_female"],
        value_vars=[col for col in ["control", "male", "female"] if col in top_features.columns],
        var_name="gender_group",
        value_name="mean_attribute_activation",
    )
    ax = long_plot.pivot(index="label", columns="gender_group", values="mean_attribute_activation").plot.barh(
        figsize=(11, max(4, 0.45 * len(top_features))),
        width=0.82,
        title=f"{language} - {profession.title()} - {trait_category.title()}: largest control/male/female differences",
    )
    ax.invert_yaxis()
    ax.set_xlabel("mean attribute-token activation")
    ax.set_ylabel("")
    ax.legend(title="")
    plt.show()
    display(top_features[["feature", "description", "spread", "control", "male", "female", "male_minus_female"]])


In [ ]:
TOP_GENDER_DISPARITY_FEATURES = 12

pair_disparities = (
    activation_records[activation_records["gender_group"].isin(["male", "female"])]
    .pivot_table(
        index=["language", "trait_category", "profession", "feature", "source", "feature_index", "description", "prompt_pair"],
        columns="gender_group",
        values="attribute_activation",
        aggfunc="max",
        fill_value=0.0,
    )
    .reset_index()
)
pair_disparities = pair_disparities[(pair_disparities.get("male", 0.0) > 0) & (pair_disparities.get("female", 0.0) > 0)].copy()
pair_disparities["male_minus_female"] = pair_disparities["male"] - pair_disparities["female"]
pair_disparities["abs_male_female_gap"] = pair_disparities["male_minus_female"].abs()

gender_disparity_tables = {}
for (language, trait_category, profession), frame in pair_disparities.groupby(["language", "trait_category", "profession"]):
    idx = frame.groupby("feature")["abs_male_female_gap"].idxmax()
    disparity = frame.loc[idx].sort_values("abs_male_female_gap", ascending=False).head(TOP_GENDER_DISPARITY_FEATURES).copy()
    disparity["label"] = disparity.apply(short_label, axis=1)
    gender_disparity_tables[(language, trait_category, profession)] = disparity

    colors = ["#1f77b4" if value >= 0 else "#d62728" for value in disparity["male_minus_female"]]
    fig, ax = plt.subplots(figsize=(11, max(4, 0.45 * len(disparity))), constrained_layout=True)
    ax.barh(disparity["label"], disparity["male_minus_female"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(f"{language} - {profession.title()} - {trait_category.title()}: largest non-zero matched male-female gaps")
    ax.set_xlabel("male minus female attribute activation on matched prompt")
    ax.set_ylabel("")
    plt.show()
    display(disparity[["feature", "description", "prompt_pair", "male_minus_female", "abs_male_female_gap", "male", "female"]])


In [ ]:
ax = group_summary.pivot_table(
    index=["language", "trait_category", "profession"],
    columns="gender_group",
    values="mean_attribute_activation",
).plot.bar(
    figsize=(11, 4.5),
    title="Mean trait-feature activation by language, profession, and gender group",
)
ax.set_xlabel("")
ax.set_ylabel("mean attribute-token activation")
ax.legend(title="")
plt.show()


## 9. Interpretation

Use `group_summary` for aggregate averages. Use `contrast_tables` for feature-level control/male/female comparisons. Use `gender_disparity_tables` for the largest non-zero matched male-female gaps.